# Download Videos from S3

In [1]:
! pip install boto3
! pip install python-dotenv


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import boto3
import os
from dotenv import load_dotenv
import pandas as pd


load_dotenv()

s3_bucket = 'deliberation-lab-recordings-contact-theory'
s3_region = "us-east-1"
recordings_folder = 'recordings'

s3 = boto3.client('s3', region_name=s3_region)

In [3]:
# download every recording from the s3 bucket

def list_all_s3_objects(bucket_name, prefix=''):
    paginator = s3.get_paginator('list_objects_v2')
    
    # Create a PageIterator from the Paginator
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)
    
    all_objects = []
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                all_objects.append(obj)
    
    return all_objects

# # without pagination just get the first 1000 objects
# response = s3.list_objects_v2(Bucket=s3_bucket)
# listings = response['Contents']

listings = list_all_s3_objects(s3_bucket)
len(listings)

2370

In [4]:
# for every file in the s3 bucket that starts with deliberation/2024
# if the directory it is in doesn't already exist, create a folder in the recordings folder
# if the file iteself doesn't exist, download the file from the s3 bucket to the folder

for obj in listings:
    if obj['Key'].startswith('deliberation/2024'):
        folder, file = obj['Key'].replace('deliberation/', "").split('/')
        if not os.path.exists(f'{recordings_folder}/{folder}'):
            print(f'Creating folder {folder}')
            os.makedirs(f'{recordings_folder}/{folder}')
        if not os.path.exists(f'{recordings_folder}/{folder}/{file}.webm'):
            print(f'Downloading {obj["Key"]}')
            s3.download_file(s3_bucket, obj['Key'], f'{recordings_folder}/{folder}/{file}.webm')

Creating folder 20241218_1636_w2s21_Q0V9QM
Creating folder 20241218_1726_w2s22_11MQMN
Creating folder 20241218_1726_w2s22_46P0MN
Creating folder 20241218_1726_w2s22_4GGZ13
Creating folder 20241218_1726_w2s22_5MFAMG
Creating folder 20241218_1726_w2s22_69VDCP
Creating folder 20241218_1726_w2s22_87CBM3
Creating folder 20241218_1726_w2s22_ABPMNH
Creating folder 20241218_1726_w2s22_JRHXCV
Creating folder 20241218_1726_w2s22_KVE3AS
Creating folder 20241218_1726_w2s22_PN3SYR
Creating folder 20241218_1726_w2s22_QSQZ74
Creating folder 20241218_1726_w2s22_RZYPHD
Creating folder 20241218_1834_w2s23_3QWKAN
Creating folder 20241218_1834_w2s23_44RMYZ
Creating folder 20241218_1834_w2s23_5FC0XS
Creating folder 20241218_1834_w2s23_6PZDCH
Creating folder 20241218_1834_w2s23_7S3XVH
Creating folder 20241218_1834_w2s23_94SH5S
Creating folder 20241218_1834_w2s23_9QY6GV
Creating folder 20241218_1834_w2s23_C599WC
Creating folder 20241218_1834_w2s23_E206N0
Creating folder 20241218_1834_w2s23_HQMVDW
Creating fo

# Measure the quantity of video we have for each respondent
We want to check and see how much video we have, accounting for reloads and technical issues. Ideally, the measure would be the number of unique seconds in which we have at least 1 (10?) frames of video. Not quite sure how to do this yet.

In [5]:
# # figure out how much of the discussion was recorded

# data = pd.read_csv("../data/formatted_data.csv")

# for i, participant in data.iterrows():
#     recordingTimes = []
#     for recordingId in participant['recordingIds']:
        


In [6]:
# ! ffprobe recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm

In [7]:
# ! ffmpeg -i recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm 2>&1 | grep "Duration"

In [8]:
# ! ffmpeg -i recordings/20240920_1732_CT_topMWAGR8/1726855356783-207f4a62-94d6-43e0-844c-2f8442334cb2-cam-audio-1726855372130.webm -vf "showinfo" -f null - 2>&1 | grep showinfo